In [2]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters.character import RecursiveCharacterTextSplitter
from langchain_text_splitters.markdown import MarkdownHeaderTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document
from langchain_chroma import Chroma

C:\Users\thaku\AppData\Local\Temp\ipykernel_3364\2171458015.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader


In [ ]:
loader_docx = Docx2txtLoader("Introduction_to_Data_and_Data_Science_2.docx")
pages = loader_docx.load()

md_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on = [("#", "Course Title"), 
                           ("##", "Lecture Title")]
)

pages_md_split = md_splitter.split_text(pages[0].page_content)
# For a typical DOCX, you'll usually have one Document.
# pages[0].page_content
# contains the entire text of the document

In [6]:
for i in range(len(pages_md_split)):
    pages_md_split[i].page_content = ' '.join(pages_md_split[i].page_content.split())
    
char_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap  = 50,
    separators=["\n\n", "\n", ". ", " ", ""]
)

pages_char_split = char_splitter.split_documents(pages_md_split)

In [4]:
embedding = OllamaEmbeddings(
    model="nomic-embed-text"
)

In [5]:
vectorstore = Chroma.from_documents(documents = pages_char_split, 
                                    embedding = embedding, 
                                    persist_directory = "./intro-to-ds-lectures")

In [6]:
vectorstore_from_directory = Chroma(persist_directory = "./intro-to-ds-lectures", 
                                    embedding_function = embedding)

In [12]:
vectorstore_from_directory

In [25]:
results = vectorstore.similarity_search(
    "What is Hadoop?",
    k=3
)

In [10]:
results

[Document(id='a75b1758-7122-4332-aaae-fe7529169d49', metadata={'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need', 'Course Title': 'Introduction to Data and Data Science'}, page_content='. It is ideal for working with mathematical functions or matrix manipulations. That’s why it is present in all categories except for ‘big data’. While respectable, MATLAB usage is a paid service, and that’s one of the reasons why it is losing ground to open-source languages like R and Python. Either way, R, Python, and MATLAB, combined with SQL, cover most of the tools used when working with traditional data, BI, and conventional data science'),
 Document(id='9efce359-2c8f-4590-9bcc-6bf2fbc04a8f', metadata={'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need', 'Course Title': 'Introduction to Data and Data Science'}, page_content='. Of course, R, and Python do have their limitations. They are not able to

In [26]:
for i in results:
    print(f"Page Content: {i.page_content}\n----------\nLecture Title:{i.metadata['Lecture Title']}\n")

Page Content: . It is ideal for working with mathematical functions or matrix manipulations. That’s why it is present in all categories except for ‘big data’. While respectable, MATLAB usage is a paid service, and that’s one of the reasons why it is losing ground to open-source languages like R and Python. Either way, R, Python, and MATLAB, combined with SQL, cover most of the tools used when working with traditional data, BI, and conventional data science
----------
Lecture Title:Programming Languages & Software Employed in Data Science - All the Tools You Need

Page Content: . Of course, R, and Python do have their limitations. They are not able to address problems specific to some domains. One example is ‘relational database management systems’—there, SQL is king. It was specifically created for that purpose. SQL is at its most advantageous when working with traditional, historical data. When preparing your BI analysis, for instance, you will surely employ it. Okay. When it comes to

In [13]:
vectorstore_from_directory.get(ids = "a75b1758-7122-4332-aaae-fe7529169d49", 
                               include = ["embeddings"])

{'ids': ['a75b1758-7122-4332-aaae-fe7529169d49'],
 'embeddings': array([[ 2.48727072e-02,  5.44272885e-02, -1.45705149e-01,
         -3.65254842e-02,  5.88424131e-02, -5.02595603e-02,
          2.04079151e-02, -5.11133745e-02, -3.81759331e-02,
         -5.77930436e-02,  6.83789887e-03,  3.94956395e-02,
          1.22609027e-01, -4.59188875e-03,  3.91964801e-03,
         -4.10034843e-02, -3.11776903e-02, -1.78744048e-02,
         -2.32260413e-02,  8.95840451e-02, -2.66406815e-02,
         -4.42944560e-03,  3.03857885e-02, -6.39013350e-02,
          7.47622326e-02, -7.40944315e-03,  2.24112123e-02,
         -5.47066680e-04, -2.64033694e-02,  4.76698242e-02,
          1.57942213e-02,  1.23254573e-02,  1.30900424e-02,
          2.64632460e-02, -9.42378119e-02, -3.14863026e-02,
          2.31381077e-02,  5.25133535e-02,  4.17127982e-02,
          1.78135058e-03, -3.60701140e-03,  1.66177247e-02,
          4.25146427e-03, -1.06397702e-03,  5.70067540e-02,
          1.48415258e-02,  4.5177289

In [14]:
vectorstore_from_directory.get("a75b1758-7122-4332-aaae-fe7529169d49")

{'ids': ['a75b1758-7122-4332-aaae-fe7529169d49'],
 'embeddings': None,
 'documents': ['. It is ideal for working with mathematical functions or matrix manipulations. That’s why it is present in all categories except for ‘big data’. While respectable, MATLAB usage is a paid service, and that’s one of the reasons why it is losing ground to open-source languages like R and Python. Either way, R, Python, and MATLAB, combined with SQL, cover most of the tools used when working with traditional data, BI, and conventional data science'],
 'uris': None,
 'included': ['metadatas', 'documents'],
 'data': None,
 'metadatas': [{'Course Title': 'Introduction to Data and Data Science',
   'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'}]}

In [16]:
added_document = Document(page_content='Alright! So… Let’s discuss the not-so-obvious differences between the terms analysis and analytics. Due to the similarity of the words, some people believe they share the same meaning, and thus use them interchangeably. Technically, this isn’t correct. There is, in fact, a distinct difference between the two. And the reason for one often being used instead of the other is the lack of a transparent understanding of both. So, let’s clear this up, shall we? First, we will start with analysis', 
                          metadata={'Course Title': 'Introduction to Data and Data Science', 
                                    'Lecture Title': 'Analysis vs Analytics'})

In [17]:
vectorstore_from_directory.add_documents([added_document])

['9dfe2c9e-8de3-4e3a-b62b-d22524c78cab']

In [18]:
vectorstore_from_directory.delete("9dfe2c9e-8de3-4e3a-b62b-d22524c78cab")

In [20]:
updated_document = Document(page_content='Great! We hope we gave you a good idea about the level of applicability of the most frequently used programming and software tools in the field of data science. Thank you for watching!', 
                            metadata={'Course Title': 'Introduction to Data and Data Science', 
                                     'Lecture Title': 'Programming Languages & Software Employed in Data Science - All the Tools You Need'})

In [21]:
vectorstore_from_directory.update_document(document_id = "55409552-1943-4892-949a-3b475ff9c840", 
                                           document = updated_document)

In [33]:
retrieved_docs = vectorstore.max_marginal_relevance_search(
    query="What software do data scientists use?", 
    k=3, 
    lambda_mult = 1, 
    filter = {"Lecture Title": "Programming Languages & Software Employed in Data Science - All the Tools You Need"}
)

In [34]:
for i in retrieved_docs:
    print(f"Page Content: {i.page_content}\n----------\nLecture Title:{i.metadata['Lecture Title']}\n")

Page Content: . Great! We hope we gave you a good idea about the level of applicability of the most frequently used programming and software tools in the field of data science. Thank you for watching!
----------
Lecture Title:Programming Languages & Software Employed in Data Science - All the Tools You Need

Page Content: . As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end
----------
Lecture Title:Programming Languages & Software Employed in Data Science - All the Tools You Need

Page Content: Alright! So… How are the techniques used in data, business intelligence, or predictive analytics appli

In [36]:
retriever = vectorstore.as_retriever(search_type = 'mmr', 
                                     search_kwargs = {'k': 3, 
                                                      'lambda_mult': 0.7})
question = "What software do data scientists use?"
retrieved_docs = retriever.invoke(question)
for i in retrieved_docs:
    print(f"Page Content: {i.page_content}\n----------\nLecture Title:{i.metadata['Lecture Title']}\n")

Page Content: . Great! We hope we gave you a good idea about the level of applicability of the most frequently used programming and software tools in the field of data science. Thank you for watching!
----------
Lecture Title:Programming Languages & Software Employed in Data Science - All the Tools You Need

Page Content: . As you can see from the infographic, R, and Python are the two most popular tools across all columns. Their biggest advantage is that they can manipulate data and are integrated within multiple data and data science software platforms. They are not just suitable for mathematical and statistical computations. In other words, R, and Python are adaptable. They can solve a wide variety of business and data-related problems from beginning to the end
----------
Lecture Title:Programming Languages & Software Employed in Data Science - All the Tools You Need

Page Content: . Among the many applications we have plotted, we can say there is an increasing amount of software de